# IoT Network Attack Detection with a Neural Network (MLP)

<a href="https://colab.research.google.com/github/buildwithyang/net-attack-predict/blob/main/notebooks/iot_attack_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Problem.** An IoT network sees thousands of traffic records ("flows"). For each one we want to
say: is it **normal** traffic or an **attack**? We train a multilayer perceptron (MLP) to decide.

**How to use this notebook.** In Colab: *Runtime → Run all*. Everything is in this one file:
it downloads the dataset by itself, and every line of data processing and training is below.
Then use the **experiment panel in Step 8** to change settings, train, and compare.

| Step | What happens | Course link |
| --- | --- | --- |
| 1 | Load and clean the data | |
| 2 | Split into train / validation / test | Week 2: generalisation, validation |
| 3 | **Data processing**: which rows, which features, how to scale | Week 2: normalisation |
| 4 | **Model**: build the MLP | Week 1: perceptron → multilayer network |
| 5 | **Training**: optimizers and the training loop | Week 1–2: backpropagation, gradient descent |
| 6 | Measure the result on the test set | confusion matrix |
| 7 | One experiment = Steps 3 → 6 in a row | |
| 8 | **Experiment panel**: Data / Model / Training tabs, train, compare | Week 2: overfitting, early stopping, dropout |
| 9 | Suggested experiments with our results | |

The three tabs of the panel match Steps 3, 4 and 5, so the work can be split: one person studies the
data processing, one the model, one the training.

## 0. Setup

Imports, and one fixed **random seed** (随机数种子). Three things in this notebook are random:
which rows go into train / validation / test, the **starting weights and biases** of the network,
and the **order of the training rows** in each epoch (*shuffling*, 打乱顺序). The seed makes all three
the same every time, so the same settings always give the same result.

In [ ]:
import copy
import time
import urllib.request
from pathlib import Path

import ipywidgets as W
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import clear_output, display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from torch import nn

SEED = 42
DATA_URL = "https://github.com/buildwithyang/net-attack-predict/raw/main/data/raw/rt-iot2022.zip"

## 1. Load and clean the data

Dataset: **RT-IoT2022** ([Kaggle](https://www.kaggle.com/datasets/supplejade/rt-iot2022real-time-internet-of-things)),
123,117 network flows, 83 features each, and a column `Attack_type` that says what the flow was.

Cleaning:
- drop `no` (just a row counter, not information about the traffic);
- remove exact duplicate rows (otherwise the same row could end up in both training and test);
- make the label: `Thing_Speak`, `MQTT_Publish`, `Wipro_bulb` are normal IoT devices → **0 = normal**;
  every other type is an attack → **1 = attack**.

In [ ]:
# Use the project copy of the data if it is there, otherwise download it (3.8 MB).
local = [Path("../data/raw/rt-iot2022.zip"), Path("data/raw/rt-iot2022.zip")]
zip_path = next((p for p in local if p.exists()), Path("rt-iot2022.zip"))
if not zip_path.exists():
    urllib.request.urlretrieve(DATA_URL, zip_path)

df = pd.read_csv(zip_path)                               # pandas can read the CSV inside the zip
df = df.drop(columns=["no"])
df["Attack_type"] = df["Attack_type"].str.strip()        # labels end with a hidden "\r" in the file
rows_before = len(df)
df = df.drop_duplicates()
print(f"{rows_before:,} rows, {rows_before - len(df):,} duplicates removed, {len(df):,} left")

NORMAL_TYPES = ["Thing_Speak", "MQTT_Publish", "Wipro_bulb"]
df["label"] = (~df["Attack_type"].isin(NORMAL_TYPES)).astype(int)   # 0 = normal, 1 = attack

counts = df.groupby("Attack_type")["label"].agg(rows="size", label="first").sort_values("rows", ascending=False)
counts["label"] = counts["label"].map({0: "normal", 1: "attack"})
display(counts)
print(f"attacks: {df['label'].mean():.1%} of all rows")

⚠️ About **90% of the rows are attacks**. A "model" that always answers *attack* would already be
90% accurate, so accuracy alone says little. We will look at two numbers instead:
- **detection rate**: of all attacks, how many did we catch? (higher is better)
- **false alarm rate**: of all normal traffic, how much did we wrongly flag as attack? (lower is better)

## 2. Split: train / validation / test

| Part | Share | Used for |
| --- | --- | --- |
| **train** | 70% | the network learns its weights from these rows |
| **validation** | 15% | checked after every epoch, to see if the network still improves on rows it did not learn from; decides when to stop |
| **test** | 15% | used **once at the very end**, for the final result |

The split is *stratified*: every attack type keeps the same share in all three parts.
It is done **once, before any data processing**, so every experiment is tested on exactly the same rows.

In [ ]:
train_df, rest = train_test_split(df, train_size=0.70, stratify=df["Attack_type"], random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=0.50, stratify=rest["Attack_type"], random_state=SEED)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {len(part):7,} rows   attacks {part['label'].mean():.1%}")

## 3. Data processing (Data tab)

A neural network only works with numbers, and the numbers should be on similar scales. Three choices,
each one a setting in the **Data** tab of the panel:

**a) Training rows.** Which training rows the network learns from. Validation and test always keep the
real mix, so results stay comparable.
- `all rows`;
- `50% / 10% / 5% / 1% of rows`: fewer rows (every traffic type keeps its share). Less data → overfits sooner;
- `undersample attacks to 1:1`: drop attack rows until there are as many attacks as normal rows
  (90% of the rows are attacks; this balances the classes).

**b) Features.** Which columns go into the network.
- `all`;
- `drop constant columns`: columns with only one value in training carry no information;
- `drop proto + service`: leave out the two text columns;
- plus **drop columns**: type any column names to leave out (e.g. features you think are noise).

`proto` and `service` are text (e.g. `tcp`, `mqtt`). **One-hot encoding** turns each value into its own 0/1 column.

**c) Scaling (normalisation).** Some columns are 0–1, some reach 20,000,000,000. Big inputs make huge
weighted sums and training becomes unstable.
- `standard`: subtract the mean, divide by the standard deviation;
- `minmax`: squeeze every column into 0 … 1;
- `robust`: like standard, but uses the median and quartiles, so extreme values matter less;
- `log + standard`: first squash huge values with a logarithm, then standardise;
- `none`: keep the raw numbers (to see why scaling matters).

Everything is **learned from the training rows only** (which columns are constant, the one-hot categories,
the mean / min / max of the scaler) and then applied to validation and test. Otherwise information from
the test set would leak into training.

In [ ]:
CATEGORICAL = ["proto", "service"]
NUMERIC = [c for c in df.columns if c not in CATEGORICAL + ["Attack_type", "label"]]
SCALERS = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler, "log + standard": StandardScaler}


def choose_training_rows(train_rows):
    if train_rows == "all rows":
        return train_df
    if train_rows == "undersample attacks to 1:1":
        normal = train_df[train_df["label"] == 0]
        attack = train_df[train_df["label"] == 1]
        share = len(normal) / len(attack)                 # keep this share of every attack type
        attack = attack.groupby("Attack_type").sample(frac=share, random_state=SEED)
        return pd.concat([normal, attack])
    fraction = float(train_rows.split("%")[0]) / 100     # "10% of rows" -> 0.1
    return train_df.groupby("Attack_type").sample(frac=fraction, random_state=SEED)


def choose_columns(train_part, features, drop_columns):
    numeric, categorical = list(NUMERIC), list(CATEGORICAL)
    if features == "drop constant columns":
        numeric = [c for c in numeric if train_part[c].nunique() > 1]
    if features == "drop proto + service":
        categorical = []
    for column in [c.strip() for c in drop_columns.split(",") if c.strip()]:
        if column not in NUMERIC + CATEGORICAL:
            raise ValueError(f"drop columns: there is no column called {column!r}")
        numeric = [c for c in numeric if c != column]
        categorical = [c for c in categorical if c != column]
    return numeric, categorical


def prepare_data(train_rows="all rows", features="all", drop_columns="", scaling="standard"):
    # Returns X (inputs) and y (labels) for train / val / test as PyTorch tensors.
    parts = {"train": choose_training_rows(train_rows), "val": val_df, "test": test_df}  # a) rows
    numeric, categorical = choose_columns(parts["train"], features, drop_columns)        # b) columns

    numbers = {name: part[numeric].to_numpy(dtype=float) for name, part in parts.items()}
    if scaling == "log + standard":
        numbers = {name: np.sign(v) * np.log1p(np.abs(v)) for name, v in numbers.items()}
    if scaling in SCALERS:                                                               # c) scaling
        scaler = SCALERS[scaling]().fit(numbers["train"])          # learned from TRAIN only
        numbers = {name: scaler.transform(v) for name, v in numbers.items()}

    onehot_columns = pd.get_dummies(parts["train"][categorical]).columns if categorical else []
    X, y = {}, {}
    for name, part in parts.items():
        blocks = [numbers[name]]
        if categorical:
            onehot = pd.get_dummies(part[categorical]).reindex(columns=onehot_columns, fill_value=0)
            blocks.append(onehot.to_numpy(dtype=float))
        X[name] = torch.tensor(np.hstack(blocks), dtype=torch.float32)
        y[name] = torch.tensor(part["label"].to_numpy(), dtype=torch.float32)
    return X, y


X, y = prepare_data()
print("X train:", tuple(X["train"].shape), "→", X["train"].shape[1], "input features per row")
# Two example columns, before and after standard scaling (training rows)
for column in ["fwd_pkts_tot", "flow_iat.tot"]:
    before, after = train_df[column], X["train"][:, NUMERIC.index(column)]
    print(f"{column:14s} before: {before.min():>8,.0f} … {before.max():>16,.0f}    after: {after.min():6.2f} … {after.max():6.2f}")

## 4. The model: a multilayer perceptron (Model tab)

Week 1 started with **one perceptron**: weighted sum of the inputs + bias → activation function.
An MLP stacks layers of them:

```
inputs (94) → hidden layer (e.g. 64 neurons, ReLU) → … → output neuron (1)
```

Settings in the **Model** tab:
- **hidden layers**: `64` = one hidden layer of 64 neurons; `128, 64` = two layers; empty = no hidden
  layer (one neuron: logistic regression). More neurons = more weights = more capacity.
- **activation** (ReLU / leaky ReLU / tanh / sigmoid) adds the non-linearity. Without it, many layers would
  be no better than one (remember XOR).
- **dropout** randomly switches off this share of neurons during training, so the network cannot rely on a
  few of them. It is a way to fight overfitting.

The last neuron gives one **score**. `sigmoid(score)` = probability that the row is an attack.

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "leaky_relu": nn.LeakyReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


def build_mlp(n_inputs, hidden_layers="64", activation="relu", dropout=0.0):
    parts = hidden_layers.replace(",", " ").split()                      # "128, 64" -> [128, 64]
    if not all(p.isdigit() and int(p) > 0 for p in parts):
        raise ValueError(f"hidden layers: use whole numbers like 64 or 128, 64 (got {hidden_layers!r})")
    hidden_sizes = [int(p) for p in parts]
    layers, width = [], n_inputs
    for size in hidden_sizes:
        layers.append(nn.Linear(width, size))           # weights + bias
        layers.append(ACTIVATIONS[activation]())         # non-linearity
        if dropout > 0:
            layers.append(nn.Dropout(dropout))           # only active while training
        width = size
    layers.append(nn.Linear(width, 1))                   # output: one score per row
    return nn.Sequential(*layers)


example = build_mlp(X["train"].shape[1])
print(example)
print("weights + biases to learn:", sum(p.numel() for p in example.parameters()))

## 5. Training (Training tab)

### 5a. The optimizer: how the weights are updated

Backpropagation gives the **gradient** of the loss for every weight (which direction makes the loss
bigger, and how fast). The **optimizer** uses it to move every weight a small step the other way.
Settings for all optimizers: **learning rate** (step size) and **weight decay** (L2 regularisation: a
*penalty* that pulls every weight a little towards 0 at each step, against overfitting).

Each optimizer also has its own settings. In the panel they appear when you pick the optimizer:

| optimizer | idea | its own settings | usual learning rate |
| --- | --- | --- | --- |
| `sgd` | plain gradient descent: `w ← w − lr × gradient` | **momentum**: keep this share of the previous step (0 = off, 0.9 is common) · **nesterov**: compute the gradient a little ahead, where momentum is taking us | 0.01 – 0.1 |
| `adam` | every weight gets its own step size, from running averages of its gradients | **beta1**: how long it remembers the gradient direction (like momentum) · **beta2**: how long it remembers the gradient size | 0.001 |
| `adamw` | Adam, but weight decay is applied directly to the weights (*decoupled*) | beta1, beta2 as Adam; use larger weight decay (e.g. 0.01) | 0.001 |
| `rmsprop` | divides each step by a running average of the gradient size | **alpha**: how long it remembers the gradient size · **momentum** | 0.001 |
| `adagrad` | divides each step by the *sum* of all past gradient sizes, so steps keep shrinking | none | 0.01 |

In the next cell each of these settings is a **dropdown** (on the right in Colab). The values you pick there are
where the panel in Step 8 starts. To change them: pick new values, run the cell, then run the panel cell again.
They start at PyTorch's defaults (`PYTORCH_DEFAULTS`); run names in the results table list every setting
that differs from these defaults.

In [ ]:
# @title Each optimizer's own settings  { display-mode: "both" }
sgd_momentum = 0             # @param [0, 0.5, 0.8, 0.9, 0.95, 0.99] {type:"raw"}
sgd_nesterov = False         # @param {type:"boolean"}
adam_beta1 = 0.9             # @param [0.5, 0.8, 0.9, 0.95, 0.99] {type:"raw"}
adam_beta2 = 0.999           # @param [0.9, 0.99, 0.999, 0.9999] {type:"raw"}
adamw_beta1 = 0.9            # @param [0.5, 0.8, 0.9, 0.95, 0.99] {type:"raw"}
adamw_beta2 = 0.999          # @param [0.9, 0.99, 0.999, 0.9999] {type:"raw"}
rmsprop_alpha = 0.99         # @param [0.9, 0.95, 0.99, 0.999] {type:"raw"}
rmsprop_momentum = 0         # @param [0, 0.5, 0.8, 0.9, 0.95, 0.99] {type:"raw"}

OPTIMIZER_SETTINGS = {           # each optimizer's own settings (the panel starts from these)
    "sgd": {"momentum": sgd_momentum, "nesterov": sgd_nesterov},
    "adam": {"beta1": adam_beta1, "beta2": adam_beta2},
    "adamw": {"beta1": adamw_beta1, "beta2": adamw_beta2},
    "rmsprop": {"alpha": rmsprop_alpha, "momentum": rmsprop_momentum},
    "adagrad": {},
}
PYTORCH_DEFAULTS = {             # PyTorch's own default values, for comparison
    "sgd": {"momentum": 0, "nesterov": False},
    "adam": {"beta1": 0.9, "beta2": 0.999},
    "adamw": {"beta1": 0.9, "beta2": 0.999},
    "rmsprop": {"alpha": 0.99, "momentum": 0},
    "adagrad": {},
}


def make_optimizer(model, optimizer, learning_rate, weight_decay, options):
    weights = model.parameters()
    if optimizer == "sgd":
        nesterov = options["nesterov"] and options["momentum"] > 0      # nesterov needs momentum
        return torch.optim.SGD(weights, lr=learning_rate, momentum=options["momentum"],
                               nesterov=nesterov, weight_decay=weight_decay)
    if optimizer == "adam":
        return torch.optim.Adam(weights, lr=learning_rate, betas=(options["beta1"], options["beta2"]),
                                weight_decay=weight_decay)
    if optimizer == "adamw":
        return torch.optim.AdamW(weights, lr=learning_rate, betas=(options["beta1"], options["beta2"]),
                                 weight_decay=weight_decay)
    if optimizer == "rmsprop":
        return torch.optim.RMSprop(weights, lr=learning_rate, alpha=options["alpha"],
                                   momentum=options["momentum"], weight_decay=weight_decay)
    if optimizer == "adagrad":
        return torch.optim.Adagrad(weights, lr=learning_rate, weight_decay=weight_decay)
    raise ValueError(f"unknown optimizer {optimizer!r}")

### 5b. The training loop

One **epoch** = one pass over all training rows, in small **batches** (**batch size** rows each).
For every batch:

1. **forward pass**: the network scores the rows;
2. **loss**: how wrong the scores are. We use *binary cross-entropy*: small when the predicted probability of
   the true class is high, large when it is low;
3. **backpropagation**: PyTorch computes the gradient of the loss for every weight (the chain rule from Week 1);
4. **update**: the optimizer changes the weights.

After each epoch we compute the **validation loss**. The lecturer's rule from Week 2: when the validation
curve stops going down and starts going up, the network is **overfitting**, so save the weights from the best
point and stop. That is **early stopping**: stop after `patience` epochs without a better validation loss, and
go back to the best weights. `patience = 0` turns it off (train all **max epochs**, keep the last weights).

In [ ]:
def train_model(model, X, y, optimizer, batch_size=256, epochs=30, patience=5):
    torch.manual_seed(SEED)
    loss_fn = nn.BCEWithLogitsLoss()                            # binary cross-entropy (sigmoid included)
    history = {"train_loss": [], "val_loss": []}
    best_val_loss, best_weights, best_epoch, waited = float("inf"), None, 0, 0
    print_every = max(1, epochs // 20)
    for epoch in range(1, epochs + 1):
        # --- train on all batches ---
        model.train()
        order = torch.randperm(len(y["train"]))                 # shuffle: new random order every epoch
        total_loss = 0.0
        for start in range(0, len(order), batch_size):
            batch = order[start : start + batch_size]
            scores = model(X["train"][batch]).squeeze(1)        # 1. forward pass
            loss = loss_fn(scores, y["train"][batch])           # 2. loss
            optimizer.zero_grad()
            loss.backward()                                     # 3. backpropagation: gradients
            optimizer.step()                                    # 4. update the weights
            total_loss += loss.item() * len(batch)
        train_loss = total_loss / len(order)

        # --- check on the validation set (no learning here) ---
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X["val"]).squeeze(1), y["val"]).item()
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        # --- early stopping: remember the best weights ---
        improved = val_loss < best_val_loss
        if improved:
            best_val_loss, best_weights, best_epoch, waited = val_loss, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            waited += 1
        if epoch % print_every == 0 or epoch == 1:
            print(f"epoch {epoch:3d}   train loss {train_loss:.4f}   val loss {val_loss:.4f}" + ("   ← best" if improved else ""))
        if patience and waited >= patience:
            print(f"early stopping at epoch {epoch}: no better val loss for {patience} epochs")
            break

    if patience and best_weights is not None:
        model.load_state_dict(best_weights)                     # go back to the best epoch
        print(f"kept the weights from epoch {best_epoch} (val loss {best_val_loss:.4f})")
    else:
        print(f"kept the last weights (epoch {epoch}); the best val loss was at epoch {best_epoch}")
    return history, best_epoch

## 6. Testing: how good is it?

The finished model scores the **test** rows (never seen during training or stopping). A probability
≥ 0.5 means "attack". The **confusion matrix** counts the four possible outcomes:

| | predicted normal | predicted attack |
| --- | --- | --- |
| **really normal** | correct | **false alarm** |
| **really attack** | **missed attack** | correct (detected) |

In [ ]:
def test_model(model, X_test, y_test, threshold=0.5):
    model.eval()
    with torch.no_grad():
        probability = torch.sigmoid(model(X_test).squeeze(1)).numpy()
    predicted = probability >= threshold
    attack = y_test.numpy() == 1
    tn, fp = int((~predicted & ~attack).sum()), int((predicted & ~attack).sum())
    fn, tp = int((~predicted & attack).sum()), int((predicted & attack).sum())
    return {"detection rate %": 100 * tp / (tp + fn), "false alarm rate %": 100 * fp / (fp + tn),
            "accuracy %": 100 * (tp + tn) / len(attack), "confusion": [[tn, fp], [fn, tp]]}


def show_run(name, history, best_epoch, result):
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.4, 1]})
    epochs = range(1, len(history["train_loss"]) + 1)
    left.plot(epochs, history["train_loss"], "--", color="gray", label="train loss")
    left.plot(epochs, history["val_loss"], color="green", label="validation loss")
    left.axvline(best_epoch, color="black", linestyle=":", label=f"best val loss (epoch {best_epoch})")
    left.set_yscale("log")
    left.set(title="Loss per epoch (lower is better)", xlabel="epoch", ylabel="loss (log scale)")
    left.legend()

    matrix = np.array(result["confusion"])
    right.imshow(matrix / matrix.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            share = matrix[i, j] / matrix[i].sum()
            right.text(j, i, f"{matrix[i, j]:,}\n{share:.2%}", ha="center", va="center",
                       color="white" if share > 0.5 else "black", fontsize=12)
    right.set_xticks([0, 1], ["normal", "attack"])
    right.set_yticks([0, 1], ["normal", "attack"])
    right.set(title="Test set: confusion matrix", xlabel="predicted", ylabel="true")
    fig.suptitle(f"{name}:  detected {result['detection rate %']:.2f}% of attacks,  "
                 f"false alarms {result['false alarm rate %']:.2f}% of normal traffic", fontweight="bold")
    fig.tight_layout()
    plt.show()

## 7. One experiment = Steps 3 → 6

An experiment is three groups of settings, one per tab of the panel. `run_experiment` runs the steps
above in order and keeps the result in `results`. The cell runs the **baseline** once; every other
experiment changes it.

In [ ]:
BASELINE = {
    "data": {"train_rows": "all rows", "features": "all", "drop_columns": "", "scaling": "standard"},
    "model": {"hidden_layers": "64", "activation": "relu", "dropout": 0.0},
    "training": {"optimizer": "adam", "learning_rate": 0.001, "weight_decay": 0.0, "options": dict(PYTORCH_DEFAULTS["adam"]),
                 "batch_size": 256, "epochs": 30, "patience": 5},
}
results = {}   # every experiment you run is kept here


def changes(settings):
    # What differs from the baseline, e.g. ["optimizer=sgd", "momentum=0.9"].
    out = []
    for group in ("data", "model", "training"):
        for key, value in settings[group].items():
            if key != "options" and value != BASELINE[group][key]:
                out.append(f"{key}={value}")
    defaults = PYTORCH_DEFAULTS[settings["training"]["optimizer"]]
    out += [f"{key}={value}" for key, value in settings["training"]["options"].items() if value != defaults[key]]
    return out


def run_experiment(name, settings):
    start = time.time()
    data, model_settings, training = settings["data"], settings["model"], settings["training"]
    X, y = prepare_data(**data)                                                       # Step 3
    torch.manual_seed(SEED)                                                           # same starting weights
    model = build_mlp(X["train"].shape[1], **model_settings)                          # Step 4
    optimizer = make_optimizer(model, training["optimizer"], training["learning_rate"],
                               training["weight_decay"], training["options"])         # Step 5a
    history, best_epoch = train_model(model, X, y, optimizer, training["batch_size"],
                                      training["epochs"], training["patience"])       # Step 5b
    result = test_model(model, X["test"], y["test"])                                  # Step 6
    results[name] = {"settings": copy.deepcopy(settings), "history": history, "best_epoch": best_epoch,
                     "train rows": len(y["train"]), "features": X["train"].shape[1],
                     "seconds": time.time() - start, **result}
    show_run(name, history, best_epoch, result)


run_experiment("baseline", BASELINE)

**How to read the loss chart** (Week 2):
- both curves go down and stay close → **good fit**;
- both stay high → **underfitting** (too small a network, too few epochs, learning rate too small …);
- train keeps going down but validation turns **up** → **overfitting**: the network learns the training
  rows by heart instead of the general pattern. The dotted line marks the best epoch.

## 8. Experiment panel ⭐

1. Change settings in the **Data**, **Model** and **Training** tabs. In Training, the settings of the chosen
   optimizer appear below it.
2. Press **Train**. The run is named after what differs from the baseline (or type your own name).
3. Under the panel: the table of all runs. **Compare** draws the validation loss of the runs you select
   (Ctrl/Cmd-click to select several).

Change **one setting at a time**, otherwise you cannot tell which change caused the difference.
The panel only collects the settings and calls `run_experiment` from Step 7; its code is folded
(*Show code* to see it).

In [ ]:
# @title Experiment panel (run this cell, then use the panel)
LEARNING_RATES = [0.00001, 0.0001, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
STYLE, WIDE = {"description_width": "110px"}, W.Layout(width="420px")


def dropdown(label, options, value):
    return W.Dropdown(options=options, value=value, description=label, style=STYLE, layout=WIDE)


def choices(label, options, value):
    return W.SelectionSlider(options=options, value=value, description=label, style=STYLE, layout=WIDE)


def note(text):
    return W.HTML(f"<div style='color:#555;max-width:640px;margin:2px 0 6px 0'>{text}</div>")


# --- Data tab ---
data_w = {
    "train_rows": dropdown("training rows", ["all rows", "50% of rows", "10% of rows", "5% of rows", "1% of rows",
                                             "undersample attacks to 1:1"], "all rows"),
    "features": dropdown("features", ["all", "drop constant columns", "drop proto + service"], "all"),
    "drop_columns": W.Text(description="drop columns", placeholder="e.g. flow_iat.tot, idle.tot", style=STYLE, layout=WIDE),
    "scaling": dropdown("scaling", ["standard", "minmax", "robust", "log + standard", "none"], "standard"),
}
data_tab = W.VBox([*data_w.values(), note("See Step 3 for what each choice does. Column names: <code>NUMERIC</code> and "
                                          "<code>CATEGORICAL</code> in Step 3.")])

# --- Model tab ---
model_w = {
    "hidden_layers": W.Text(description="hidden layers", placeholder="64  or  128, 64  (empty = none)", style=STYLE, layout=WIDE),
    "activation": dropdown("activation", list(ACTIVATIONS), "relu"),
    "dropout": W.FloatSlider(min=0, max=0.7, step=0.05, description="dropout", style=STYLE, layout=WIDE),
}
model_tab = W.VBox([*model_w.values(), note("See Step 4.")])

# --- Training tab: shared settings + the chosen optimizer's own settings ---
train_w = {
    "optimizer": dropdown("optimizer", list(OPTIMIZER_SETTINGS), "adam"),
    "learning_rate": choices("learning rate", LEARNING_RATES, 0.001),
    "weight_decay": choices("weight decay", [0.0, 0.00001, 0.0001, 0.001, 0.01, 0.1], 0.0),
    "batch_size": choices("batch size", [16, 32, 64, 128, 256, 512, 1024, 4096], 256),
    "epochs": W.IntSlider(min=1, max=200, value=30, description="max epochs", style=STYLE, layout=WIDE),
    "patience": W.IntSlider(min=0, max=20, value=5, description="patience", style=STYLE, layout=WIDE),
}
MOMENTUMS, BETA1S, BETA2S, ALPHAS = [0, 0.5, 0.8, 0.9, 0.95, 0.99], [0.5, 0.8, 0.9, 0.95, 0.99], [0.9, 0.99, 0.999, 0.9999], [0.9, 0.95, 0.99, 0.999]
start = OPTIMIZER_SETTINGS   # the values picked in Step 5a
option_w = {   # one set of widgets per optimizer, so each keeps its own values
    "sgd": {"momentum": choices("momentum", MOMENTUMS, start["sgd"]["momentum"]),
            "nesterov": W.Checkbox(value=start["sgd"]["nesterov"], description="nesterov (needs momentum > 0)", indent=False)},
    "adam": {"beta1": choices("beta1", BETA1S, start["adam"]["beta1"]),
             "beta2": choices("beta2", BETA2S, start["adam"]["beta2"])},
    "adamw": {"beta1": choices("beta1", BETA1S, start["adamw"]["beta1"]),
              "beta2": choices("beta2", BETA2S, start["adamw"]["beta2"])},
    "rmsprop": {"alpha": choices("alpha", ALPHAS, start["rmsprop"]["alpha"]),
                "momentum": choices("momentum", MOMENTUMS, start["rmsprop"]["momentum"])},
    "adagrad": {},
}
option_notes = {
    "sgd": "w ← w − lr × gradient. Momentum keeps part of the previous step. Usual lr 0.01–0.1.",
    "adam": "Own step size per weight. beta1 remembers the direction, beta2 the size. Usual lr 0.001.",
    "adamw": "Adam with decoupled weight decay; try weight decay 0.01. Usual lr 0.001.",
    "rmsprop": "Divides each step by a running average of the gradient size (alpha = how long it remembers). Usual lr 0.001.",
    "adagrad": "No extra settings: each weight's steps shrink as its gradients add up. Usual lr 0.01.",
}
option_box = W.VBox()


def show_optimizer_settings(*_):
    name = train_w["optimizer"].value
    option_box.children = [W.HTML(f"<b>Settings of {name}</b>"), *option_w[name].values(), note(option_notes[name])]


train_w["optimizer"].observe(show_optimizer_settings, names="value")
show_optimizer_settings()
training_tab = W.VBox([W.HBox([W.VBox([train_w[k] for k in ("optimizer", "learning_rate", "weight_decay")]),
                               W.VBox([train_w[k] for k in ("batch_size", "epochs", "patience")])]),
                       option_box, note("See Step 5.")])

tabs = W.Tab(children=[data_tab, model_tab, training_tab])
for i, title in enumerate(["Data", "Model", "Training"]):
    tabs.set_title(i, title)


def read_panel():
    training = {k: w.value for k, w in train_w.items()}
    training["options"] = {k: w.value for k, w in option_w[training["optimizer"]].items()}
    return {"data": {k: w.value for k, w in data_w.items()},
            "model": {k: w.value for k, w in model_w.items()},
            "training": training}


def set_panel(settings, with_options=True):
    for group, widgets in (("data", data_w), ("model", model_w), ("training", train_w)):
        for key, widget in widgets.items():
            widget.value = settings[group][key]
    if with_options:
        for key, value in settings["training"]["options"].items():
            option_w[settings["training"]["optimizer"]][key].value = value


def unique(name):
    new, n = name, 2
    while new in results:
        new, n = f"{name} #{n}", n + 1
    return new


name_w = W.Text(description="run name", placeholder="(empty = named after the changes)", style=STYLE, layout=WIDE)
train_btn = W.Button(description="Train", button_style="success", icon="play")
reset_btn = W.Button(description="Reset to baseline", icon="undo")
status, run_out = W.HTML(), W.Output()


def on_train(_):
    settings = read_panel()
    name = unique(name_w.value.strip() or ", ".join(changes(settings)) or "baseline")
    train_btn.disabled = True
    status.value = f"training <b>{name}</b> …"
    run_out.clear_output()
    error = None
    with run_out:
        try:
            run_experiment(name, settings)
        except Exception as problem:
            error = problem
    train_btn.disabled = False
    if error:
        status.value = f"<span style='color:#c0392b'><b>error:</b> {error}</span>"
    else:
        status.value = f"done: <b>{name}</b> in {results[name]['seconds']:.1f} s"
        refresh_results()


train_btn.on_click(on_train)
reset_btn.on_click(lambda _: (set_panel(BASELINE), setattr(name_w, "value", "")))

# --- results table + compare ---
table_out, compare_out = W.Output(), W.Output()
pick = W.SelectMultiple(description="runs", rows=6, style=STYLE, layout=W.Layout(width="640px"))
compare_btn = W.Button(description="Compare selected", icon="bar-chart")


def results_table():
    rows = {name: {"changes vs baseline": ", ".join(changes(r["settings"])) or "-",
                   "train rows": r["train rows"], "features": r["features"],
                   "epochs run": len(r["history"]["val_loss"]), "best epoch": r["best_epoch"],
                   "best val loss": round(min(r["history"]["val_loss"]), 4),
                   "detected %": round(r["detection rate %"], 2), "false alarms %": round(r["false alarm rate %"], 2),
                   "seconds": round(r["seconds"], 1)}
            for name, r in results.items()}
    return pd.DataFrame(rows).T


def refresh_results():
    pick.options = list(results)
    with table_out:
        clear_output(wait=True)
        display(results_table())


def on_compare(_):
    with compare_out:
        clear_output(wait=True)
        names = list(pick.value) or list(results)
        plt.figure(figsize=(10, 4.5))
        for name in names:
            val_loss = results[name]["history"]["val_loss"]
            plt.plot(range(1, len(val_loss) + 1), val_loss, label=name)
        plt.yscale("log")
        plt.title("Validation loss per epoch (lower is better)")
        plt.xlabel("epoch")
        plt.ylabel("val loss (log scale)")
        plt.legend()
        plt.show()
        display(results_table().loc[names])


compare_btn.on_click(on_compare)
set_panel(BASELINE, with_options=False)   # optimizer settings keep the values picked in Step 5a
refresh_results()
display(W.VBox([tabs, W.HBox([name_w, train_btn, reset_btn]), status, run_out,
                W.HTML("<h3>All runs</h3>"), table_out, W.HBox([pick, compare_btn]), compare_out]))

## 9. Suggested experiments

Each experiment changes **one** thing in the panel, starting from the baseline (*Reset to baseline*).
**Guess first** (先猜再跑), then train, then compare with the numbers below. The groups follow the three
tabs, so each person can take one.

"We saw" = one run on a laptop CPU, seed 42. Colab may differ in the last digits; the trend is what
matters. Only one seed was run, so small differences (e.g. 1.3% vs 1.4% false alarms) may be random.

**Baseline**: all rows, standard scaling, hidden `64`, ReLU, Adam lr 0.001, batch 256, patience 5 →
best val loss **0.0076**, attacks detected **99.84%**, false alarms **1.28%**.

### Data tab: how does data processing change the result?

| change | best val loss | detected | false alarms | note |
| --- | --- | --- | --- | --- |
| scaling `none` | 7.3 | 96.6% | 1.2% | first-epoch loss ≈ 45,000: huge inputs → huge weighted sums |
| scaling `robust` | 3.3 | 98.2% | 5.3% | also unstable. Probably: in many columns the middle 50% of values are nearly equal, so dividing by that tiny range blows values up |
| scaling `minmax` | 0.017 | 99.8% | 2.8% | stable but worse. Probably: the huge values set the max, so most rows get squeezed near 0 |
| scaling `log + standard` | **0.0058** | 99.8% | **0.9%** | best: the log tames the 2e10 values first |
| features `drop constant columns` | 0.0083 | 99.8% | 1.4% | 93 features; a constant column carries no information, so ~no change |
| features `drop proto + service` | 0.0096 | 99.8% | 2.2% | 81 features; the protocol / service columns help |
| drop columns `flow_iat.tot, fwd_iat.tot, bwd_iat.tot, idle.tot` | 0.0094 | 99.8% | 1.4% | the four largest-valued columns: small loss |
| training rows `undersample attacks to 1:1` | 0.031 | 98.7% | **1.1%** | 16,821 rows. Fewer false alarms, more missed attacks. Val loss is not comparable here: validation keeps the real 90% attacks |
| training rows `10% of rows` | 0.042 | 99.6% | 5.0% | 8,254 rows: less data, worse model |
| training rows `1% of rows` | 0.11 | 97.7% | 6.0% | 826 rows |

### Model tab: how does the network change the result?

| change | weights | best val loss | false alarms | note |
| --- | --- | --- | --- | --- |
| hidden layers empty (one neuron, logistic regression) | 95 | 0.055 | 7.5% | no hidden layer: a straight-line boundary |
| hidden `8` | 769 | 0.017 | 2.4% | |
| hidden `64` (baseline) | 6,145 | 0.0076 | 1.3% | |
| hidden `128, 64` | 20,481 | 0.0072 | 0.8% | early stopping at epoch 20 |
| hidden `256, 128` | 57,345 | 0.0074 | 0.8% | 3× the weights, no better: capacity is not the limit any more |
| activation `tanh` / `leaky_relu` | | 0.0083 / 0.0082 | 1.3% / 1.3% | about the same as ReLU |
| activation `sigmoid` | | 0.0125 | 2.0% | slower |
| dropout `0.2` / `0.5` | | 0.0097 / 0.013 | 1.7% / 2.2% | with all rows there is little overfitting, so dropout only slows learning |

**Vanishing gradients:** hidden `64, 64, 64, 64` + optimizer `sgd`, lr `0.01`. With `sigmoid` the loss stays
at 0.33 for all 30 epochs and the model calls **every** row an attack (100% false alarms). The sigmoid's
slope is at most 0.25 and backpropagation multiplies it once per layer, so the early layers hardly change.
With `relu`: val loss 0.026, false alarms 3.3%.

### Training tab: optimizers and their settings

**Learning rate** (Adam):

| lr | best val loss | false alarms | note |
| --- | --- | --- | --- |
| 0.00001 | 0.113 | 35% | too small: still learning after 30 epochs |
| 0.0001 | 0.028 | 5.5% | slow |
| 0.001 (baseline) | 0.0076 | 1.3% | |
| 0.01 | 0.0097 | 1.8% | fast at first, early stopping at epoch 24 |
| 0.1 | 0.061 | 16% | too big: val loss jumps around, stops at epoch 7 |

**SGD and momentum** (the learning rate matters much more for SGD):

| optimizer settings | best val loss | false alarms |
| --- | --- | --- |
| sgd, lr 0.001 | 0.100 | 35% |
| sgd, lr 0.01 | 0.047 | 6.7% |
| sgd, lr 0.01, momentum 0.9 | 0.017 | 2.3% |
| sgd, lr 0.01, momentum 0.9, nesterov | 0.018 | 2.0% |
| sgd, lr 0.1 | 0.017 | 2.1% |
| sgd, lr 0.1, momentum 0.9 | **0.0096** | 1.8% |

Momentum 0.9 makes lr 0.01 behave about like lr 0.1 without momentum: past steps add up.

**Adam, AdamW, RMSprop, Adagrad** (lr 0.001 unless written):

| optimizer settings | best val loss | false alarms | note |
| --- | --- | --- | --- |
| adam (baseline: beta1 0.9, beta2 0.999) | 0.0076 | 1.3% | |
| adam, beta1 0.5 / 0.99 | 0.0079 / 0.0086 | 1.4% / 1.6% | beta1 matters little here |
| adam, beta2 0.9 | 0.0117 | 2.0% | short memory of the gradient size → noisier steps, stops at epoch 21 |
| adamw, weight decay 0.01 | 0.0078 | 1.3% | same as Adam here (no overfitting to fight) |
| rmsprop (alpha 0.99) | **0.0075** | 1.3% | as good as Adam |
| rmsprop, alpha 0.9 | 0.015 | 2.0% | |
| rmsprop, momentum 0.9 | 0.012 | 2.0% | very fast start, stops at epoch 14 |
| adagrad, lr 0.01 | 0.019 | 2.8% | steps keep shrinking |
| adagrad, lr 0.001 | 0.068 | 5.2% | too slow |

**Batch size**: steps per epoch = 82,545 ÷ batch size. Batch `32` (2,580 steps per epoch): best val loss
0.0080 but about 7× slower per epoch. Batch `4096` (21 steps per epoch): only 0.034 after 30 epochs,
false alarms 5.9%.

### Together: make it overfit, then fix it ⭐ (Week 2)

With all rows this network hardly overfits. To *see* overfitting, combine the three tabs:

> Data: training rows `1% of rows` (826 rows) · Model: hidden `256, 128` · Training: max epochs `150`, patience `0`

We saw: train loss falls to 0.0015, but **validation loss is lowest at epoch 7 (0.106) and then climbs to
0.31**. The network learns the 826 rows by heart. Keep these settings and change one thing:

| fix | tab | val loss at the end | note |
| --- | --- | --- | --- |
| nothing | | 0.31 | overfitting |
| patience `5` (early stopping) | Training | stops at epoch 12, keeps epoch 7: **0.106** | the lecturer's rule: keep the best weights, stop |
| weight decay `0.001` / `0.01` | Training | 0.16 / **0.12** | the penalty keeps weights small |
| dropout `0.3` / `0.5` | Model | 0.29 / 0.28 | barely helps here |
| hidden `16` (smaller network) | Model | 0.21 (lowest 0.116 at epoch 59) | overfits later and less |
| training rows `10% of rows` | Data | 0.089 (lowest 0.031 at epoch 13) | 10× more data: much lower loss |

### For the report

For each experiment: the table and the validation loss chart from the panel (*Compare selected*),
one sentence on what changed, and one sentence on *why* (the course idea behind it).